In [127]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import matplotlib.pyplot as plt


In [128]:
# #!/bin/bash
# !kaggle datasets download preetviradiya/english-hindi-dataset

In [129]:
# !unzip english-hindi-dataset.zip

In [130]:
import pandas as pd

In [131]:
df = pd.read_csv("Dataset_English_Hindi.csv")

In [132]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130476 entries, 0 to 130475
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   English  130474 non-null  object
 1   Hindi    130164 non-null  object
dtypes: object(2)
memory usage: 2.0+ MB


In [133]:
df.head()

,English,Hindi
0,Help!,बचाओ!
1,Jump.,उछलो.
2,Jump.,कूदो.
3,Jump.,छलांग.
4,Hello!,नमस्ते।


In [134]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130476 entries, 0 to 130475
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   English  130474 non-null  object
 1   Hindi    130164 non-null  object
dtypes: object(2)
memory usage: 2.0+ MB


In [135]:
df["English"] = df["English"].str.lower().str.split()
df["Hindi"] = df["Hindi"].str.split()

In [136]:
df.head()

,English,Hindi
0,[help!],[बचाओ!]
1,[jump.],[उछलो.]
2,[jump.],[कूदो.]
3,[jump.],[छलांग.]
4,[hello!],[नमस्ते।]


In [137]:
df["Hindi"] = df["Hindi"].apply(
    lambda x: ["<start>"] + x + ["<end>"] if isinstance(x, list) else ["<start>", "<end>"]
    )

In [138]:
df.head()

,English,Hindi
0,[help!],"[<start>, बचाओ!, <end>]"
1,[jump.],"[<start>, उछलो., <end>]"
2,[jump.],"[<start>, कूदो., <end>]"
3,[jump.],"[<start>, छलांग., <end>]"
4,[hello!],"[<start>, नमस्ते।, <end>]"


In [139]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130476 entries, 0 to 130475
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   English  130474 non-null  object
 1   Hindi    130476 non-null  object
dtypes: object(2)
memory usage: 2.0+ MB


In [140]:
df_subset = df.head(20000)

In [141]:

df_subset = df_subset.dropna(subset=['English', 'Hindi'])

vocab_english = []
for i , sentance in enumerate(df_subset["English"]) :
  if isinstance(sentance, list):
    vocab_english.extend(sentance)

vocab_english = list(set(vocab_english))
vocab_english = ["<PAD>" , "<UNK>"] + vocab_english
print(f"English Vocab Size: {len(vocab_english)}")

English Vocab Size: 33089


In [142]:
vocab_hindi = []
for i , sentance in enumerate(df_subset["Hindi"]) :
  vocab_hindi.extend(sentance)

vocab_hindi = list(set(vocab_hindi))
vocab_hindi = ["<PAD>" , "<start>" , "<end>"] + vocab_hindi
print(vocab_hindi)
print(len(vocab_hindi))


['<PAD>', '<start>', '<end>', 'तरह-एकल', "they're", '1914', 'संक्रमित', 'है-उसे', 'भूमि', 'दुर्घटनाओं', 'मंड़पम', 'बन्ने', 'मेलाइक', 'कोट्टु', 'विद्युत-आपूर्ति', 'रणनीतिकार', 'अमरीकी', 'सेकडों', 'गीली,', 'समस्योओं', 'न?”', '(biological', 'ड़ी.वूल्फेनसोन', 'सामंती', 'नाविक', 'अधिमिश्रण', 'यादृच्छिक', 'एपीजे', 'खानदानी', 'वाहिनी', 'वऋ-ऊण्श्छ्ष्-यतीत', 'द्विभाषीय', 'खिलाये', 'पुष्पिकाएं', 'जला', 'किसको', 'ज्ञानांकुर', 'शांतनु', 'हेटेरॉप्टेरा', 'बेशुमार', 'चढ़े', 'अपर्याप्तता', 'मापन', 'उठा', 'गड्डिंयां', 'डोंब', 'गुंथे', 'भविषऋ-ऊण्श्छ्ष्-य', 'घेरा', 'पांचवें', 'हड्डियां', 'कोंड़ोलीज़ा', 'बट', 'कताई', 'बढ़ना', 'क़दमों', 'मुग़ल', 'नागिदेव', 'मिलेंगे।', 'संकल्पों', 'उड़ातें', 'चुम्बकीय', 'मुकझा', 'हिमनदों', 'मुखों', 'Ka%aa', 'वाकर', 'Anthropogenic', 'चालुकयों', 'आलकमान', 'क्षतिग्रस्त', 'है.धीरे', 'केशे', 'फूंकने', 'चालुक़्यों', 'चौबीस', 'छू', 'केद्रित', 'नादुरूस्त', 'सिलसिले', 'सेनापतियों', 'अभिव्यक्तियां', 'सुनिए,', 'हो.', 'अध्यापक', 'सा', 'वाशिंगटन', 'कोमपीनयों', 'उल्फा', 'DEFRA', 'भारीमात

In [143]:
eng_word_2_idx = {word : idx for idx , word in enumerate(vocab_english)}
print(eng_word_2_idx)

{'<PAD>': 0, '<UNK>': 1, 'timely': 2, "they're": 3, 'medieval': 4, '1914': 5, 'future,and': 6, 'initiatives': 7, 'animals.': 8, 'transformation.': 9, 'vasantesvaram': 10, 'regroup': 11, 'discount': 12, 'upto': 13, 'impersonal': 14, '80th': 15, 'smelled': 16, 'rewards': 17, 'spur': 18, 'smitha-who': 19, 'chakor-chakon': 20, 'occupy': 21, 'announced,': 22, '(moksham).': 23, 'hg': 24, 'extricate': 25, 'inspiring': 26, 'nuhari': 27, 'asi': 28, '1518': 29, 'politely': 30, 'vaca': 31, '(1974)': 32, 'pictures.': 33, 'proves': 34, 'got.': 35, '350km': 36, 'exile.': 37, 'kunduz': 38, 'parvana(1971)': 39, 'makeover': 40, 'sharing.': 41, 'religion.in': 42, 'sudhra': 43, 'airplane': 44, 'initiated': 45, 'weaknesses': 46, 'meter': 47, 'karachi': 48, 'companies,': 49, 'tawa.': 50, 'starve.': 51, 'equal.': 52, 'subjudice': 53, 'wesley,': 54, 'fifty': 55, 'scepticism': 56, 'jainisru': 57, 'restaurants': 58, 'or,': 59, 'toss.': 60, 'maniacs': 61, 'xp-group': 62, 'trained': 63, 'denomination': 64, '(isa

In [144]:
hindi_word_2_idx = {word : idx for idx , word in enumerate(vocab_hindi)}
print(hindi_word_2_idx)

{'<PAD>': 0, '<start>': 26242, '<end>': 30857, 'तरह-एकल': 3, "they're": 4, '1914': 5, 'संक्रमित': 6, 'है-उसे': 7, 'भूमि': 8, 'दुर्घटनाओं': 9, 'मंड़पम': 10, 'बन्ने': 11, 'मेलाइक': 12, 'कोट्टु': 13, 'विद्युत-आपूर्ति': 14, 'रणनीतिकार': 15, 'अमरीकी': 16, 'सेकडों': 17, 'गीली,': 18, 'समस्योओं': 19, 'न?”': 20, '(biological': 21, 'ड़ी.वूल्फेनसोन': 22, 'सामंती': 23, 'नाविक': 24, 'अधिमिश्रण': 25, 'यादृच्छिक': 26, 'एपीजे': 27, 'खानदानी': 28, 'वाहिनी': 29, 'वऋ-ऊण्श्छ्ष्-यतीत': 30, 'द्विभाषीय': 31, 'खिलाये': 32, 'पुष्पिकाएं': 33, 'जला': 34, 'किसको': 35, 'ज्ञानांकुर': 36, 'शांतनु': 37, 'हेटेरॉप्टेरा': 38, 'बेशुमार': 39, 'चढ़े': 40, 'अपर्याप्तता': 41, 'मापन': 42, 'उठा': 43, 'गड्डिंयां': 44, 'डोंब': 45, 'गुंथे': 46, 'भविषऋ-ऊण्श्छ्ष्-य': 47, 'घेरा': 48, 'पांचवें': 49, 'हड्डियां': 50, 'कोंड़ोलीज़ा': 51, 'बट': 52, 'कताई': 53, 'बढ़ना': 54, 'क़दमों': 55, 'मुग़ल': 56, 'नागिदेव': 57, 'मिलेंगे।': 58, 'संकल्पों': 59, 'उड़ातें': 60, 'चुम्बकीय': 61, 'मुकझा': 62, 'हिमनदों': 63, 'मुखों': 64, 'Ka%aa': 65, 'वाकर': 6

In [145]:
def word_to_number(sentance , vocab):
  value = []
  for i in sentance:
    value.append(vocab[i])
  return value

In [146]:
word_to_number(["reminded" , "and" , "in"] , eng_word_2_idx)

[18430, 25460, 31066]

In [147]:
# ["English"] , df_subset["hindi"]

In [148]:
temp_for_eng = []
temp_for_hindi = []
for i , j in zip(df_subset["English"] , df_subset["Hindi"]):
  temp_for_eng.append(word_to_number(i , eng_word_2_idx))
  temp_for_hindi.append(word_to_number(j , hindi_word_2_idx))


# print(temp_for_eng)
df_subset["English"] = temp_for_eng
df_subset["Hindi"] = temp_for_hindi

In [149]:
df_subset["English"][100]
df_subset["Hindi"][100]

[26242, 23727, 23555, 12947, 31790, 22851, 30857]

In [150]:
df_subset.head()

,English,Hindi
0,[9290],"[26242, 7127, 30857]"
1,[23804],"[26242, 19125, 30857]"
2,[23804],"[26242, 6100, 30857]"
3,[23804],"[26242, 1159, 30857]"
4,[29185],"[26242, 32444, 30857]"


In [153]:
from torch.utils.data import DataLoader , Dataset
import torch

class dataset(Dataset):
  def __init__(self , df):
    self.x = df["English"].tolist()
    self.y = df["Hindi"].tolist()

  def __len__(self):
    return len(self.x)

  def __getitem__(self , index):
    return torch.tensor(self.x[index] , dtype=torch.long) , torch.tensor(self.y[index] , dtype=torch.long)

In [154]:
from torch.nn.utils.rnn import pad_sequence
import torch

PAD_IDX = 0

def collate_fn(batch):

    src = [x[0] for x in batch]
    trg = [x[1] for x in batch]

    src = pad_sequence(
        src,
        batch_first=True,
        padding_value=PAD_IDX
    )

    trg = pad_sequence(
        trg,
        batch_first=True,
        padding_value=PAD_IDX
    )

    return src, trg

In [155]:
train_dataset = dataset(df_subset)
dataLoader = DataLoader(train_dataset , batch_size=8 , shuffle=True , collate_fn=collate_fn)

In [156]:
for i , (m, n) in enumerate(dataLoader):
  print(m.size())
  if i == 5 : break

torch.Size([8, 18])
torch.Size([8, 43])
torch.Size([8, 14])
torch.Size([8, 40])
torch.Size([8, 22])
torch.Size([8, 18])


In [157]:
class Encoder(nn.Module):
  def __init__(self , vocab_size):
    super().__init__()
    self.emb = nn.Embedding(vocab_size , 128)
    self.GRU = nn.GRU(128 , 50 , 3 , batch_first=True)

  def forward(self , x):
    emb = self.emb(x)
    output , hidden = self.GRU(emb)
    return hidden

In [158]:
class Decoder(nn.Module):
  def __init__(self , vocab_size):
    super().__init__()
    self.emb = nn.Embedding(vocab_size , 128)
    self.gru = nn.GRU(128 , 50 , 3 , batch_first=True)
    self.fc = nn.Linear(50 , vocab_size)

  def forward(self , prev_prediction , hidden):
    emb = self.emb(prev_prediction)
    emb = emb.unsqueeze(1)

    output , hidden = self.gru(emb , hidden)
    output = output.squeeze(1)

    prediction = self.fc(output)

    return prediction , hidden

In [159]:
len(vocab_english)

33089

In [160]:
class Seq2Seq(nn.Module):
  def __init__(self , vocab_english , vocab_hindi):
    super().__init__()
    self.encoder = Encoder(len(vocab_english))
    self.decoder = Decoder(len(vocab_hindi))

  def forward(self , english_src , hindi_target):
    hidden = self.encoder(english_src)
    decoder_input = hindi_target[: , 0]

    predictions = []

    target_len = hindi_target.shape[1]

    for t in range(1, target_len):

      prediction , hidden = self.decoder(decoder_input , hidden)
      predictions.append(prediction)

      teacher_forcing_ratio = 0.7
      teacher_force = random.random() < teacher_forcing_ratio

      if teacher_force:
        decoder_input = hindi_target[: , t]
      else:
        decoder_input = prediction.argmax(dim=1)

    predictions = torch.stack(predictions, dim=1)
    return predictions

In [161]:
# import torch_xla.core.xla_model as xm
# device = xm.xla_device()

In [162]:
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
device

device(type='cuda')

In [163]:
model = Seq2Seq(vocab_english , vocab_hindi)
model = model.to(device)

In [164]:
epochs = 2
learning_rate = 0.001

In [165]:
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(params=model.parameters() , lr=learning_rate)

In [ ]:
losses = []
for epoch in range(epochs):
  total_loss = 0
  for i , (src , target) in enumerate(dataLoader):
    optimizer.zero_grad()
    src = src.to(device)
    target = target.to(device)
    output = model(src , target)

    loss = loss_fn(output.reshape(-1, len(vocab_hindi)),target[:, 1:].reshape(-1))
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    total_loss += loss.item()
    if i % 4000 == 0:
      print("1", end="--", flush=True)

    avg_loss = total_loss / len(dataLoader)
    losses.append(avg_loss)
  print()
  print(f"Epoch: {epoch+1} , Loss: {avg_loss:.4f}")


1--

In [ ]:
plt.plot(range(1, epochs + 1), losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss per Epoch")
plt.grid(True)

In [ ]:
torch.save(model.state_dict() , f"model_epoch{epochs}.pth")
torch.save(model.state_dict(), 'model.pth')

In [ ]:
del model
del optimizer
del output
del loss
import gc
gc.collect()
torch.cuda.empty_cache()